In [2]:
using Pkg
Pkg.activate("/Users/jkarpowski/Dendron/notes/assets/scripts/")
using SparseArrays
using LinearAlgebra
using BenchmarkTools

using Revise
using Arrhenius

  Activating project at `~/Dendron/notes/assets/scripts`


In [3]:
abstract type Reactions end

struct ForwardReactions{Ty<:AbstractArray,sTy<:AbstractArray}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    vk::sTy
end

struct ReversibleReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_sum::vTy
end

struct ThreeBodyReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_sum::vTy
    efficiency_coeffs::sTy
end

In [62]:
gas = CreateSolution("../mechanism/gri30.yaml",sparseCoeffs=true);
kf = zeros(gas.n_reactions);
kr = zeros(gas.n_reactions);
C = rand(gas.n_species)
C = C./sum(C)
wdot = zeros(gas.n_species)
T =900

900

In [5]:
forwardReactions = ForwardReactions(gas.kinetics.Arrhenius_coeffs,
                    gas.kinetics.reactant_orders,
                    sparse(transpose(gas.kinetics.vk))
                    )

reversibleReactions = ReversibleReactions(
    gas.kinetics.Arrhenius_coeffs,
    gas.kinetics.reactant_orders,
    gas.kinetics.product_stoich_coeffs,
    gas.kinetics.reactant_stoich_coeffs,
    sparse(transpose(gas.kinetics.vk)),
    gas.kinetics.vk_sum
    );

i_tB = gas.kinetics.index_three_body
threeBodyReactions = ThreeBodyReactions(
    gas.kinetics.Arrhenius_coeffs[i_tB,:],
    gas.kinetics.reactant_orders[:,i_tB],
    gas.kinetics.product_stoich_coeffs[:,i_tB],
    gas.kinetics.reactant_stoich_coeffs[:,i_tB],
    sparse(transpose(gas.kinetics.vk[:,i_tB])),
    gas.kinetics.vk_sum[i_tB],
    gas.kinetics.efficiencies_coeffs[:,i_tB]
    );

In [6]:



function forwardRate!(kf::AbstractVector, arr_coeffs::AbstractArray,T::Real)
    logT = log(T)
    Eexp = (4184.0 / Arrhenius.R / T)
    for i in eachindex(kf) 
        @inbounds  kf[i] = arr_coeffs[i, 1]  * exp(arr_coeffs[i, 2] * logT - arr_coeffs[i, 3] * Eexp)
    end
    #@. kf = @view(arr_coeffs[:, 1])  * exp(@view(arr_coeffs[:, 2]) * logT - @view(arr_coeffs[:, 3]) * Eexp)
end
function kf_Pi_c_rectantOrder!(kf,C,reactant_orders)
    for i in eachindex(kf)
        for j in reactant_orders.colptr[i]:reactant_orders.colptr[i+1]-1
            @inbounds kf[i] *= C[reactant_orders.rowval[j]]^reactant_orders.nzval[j]
        end
    end
end
function vk_kf!(wdot,kf,vk) # TODO use likwid to compare the two methods
    # transposed vk - workflow 
    for isp in eachindex(wdot) 
        for j in vk.colptr[isp]:vk.colptr[isp+1]-1
            wdot[isp] += vk.nzval[j] * kf[vk.rowval[j]]
        end
    end 
    # for ir in eachindex(kf)
    #     for j in vk.colptr[ir]:vk.colptr[ir+1]-1
    #         wdot[vk.rowval[j]] += vk.nzval[j] * kf[ir]
    #     end
    # end
    #mul!(wdot,vk',kf)
    return nothing
end
function wdot_func!(wdot::AbstractVector,kf::AbstractVector,C::AbstractVector,forwardReactions::ForwardReactions ,T::Real)
    forwardRate!(kf, forwardReactions.arrhenius_coeffs,T)
    kf_Pi_c_rectantOrder!(kf,C,forwardReactions.reactant_orders)
    vk_kf!(wdot,kf,forwardReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$C,$forwardReactions,$T) evals = 200 seconds = 10

BenchmarkTools.Trial: 5938 samples with 200 evaluations.
 Range (min … max):  6.527 μs … 34.500 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     8.050 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   8.354 μs ±  1.239 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

           █                                                  
  ▂▂▁▂▁▁▂▁▂██▅▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂ ▃
  6.53 μs        Histogram: frequency by time        12.7 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

# Comparison to old wdot

In [7]:
function wdot_func(reaction, T, C)

    @inbounds _kf = @. @view(reaction.Arrhenius_coeffs[:, 1]) * exp(
        @view(reaction.Arrhenius_coeffs[:, 2]) * log(T) -
        @view(reaction.Arrhenius_coeffs[:, 3]) * (4184.0 / R / T),
    )
    for i = 1:reaction.n_reactions
        @inbounds for j in reaction.i_reactant[i]
            @inbounds _kf[i] *= C[j]^reaction.reactant_orders[j, i]
        end
    end
    

    #if get_qdot
    #    return _kf - _kr
    #end
    return reaction.vk *  (_kf ) #omega_dot_k
end

@benchmark wdot_func($(gas.kinetics),$T,$C)

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  16.037 μs … 171.351 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     16.419 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   18.292 μs ±   7.425 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▅▃▄▂                                                        ▁
  ██████▇▆▆▇▇▇▇▅▆▆▆▅▄▅▄▅▅▅▅▄▅▅▄▅▆▅▆▅▆▅▅▅▅▅▆▅▅▅▅▆▆▅▆▅▄▅▅▄▄▅▅▅▄▅ █
  16 μs         Histogram: log(frequency) by time      50.1 μs <

 Memory estimate: 3.17 KiB, allocs estimate: 2.

In [8]:
wdot .=0.0
wdot_func!(wdot,kf,C,forwardReactions,T)
wdot-wdot_func((gas.kinetics),T,C)

53-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

# reversibleReaction

In [9]:
h_RT = rand(gas.n_species)
s0_R = rand(gas.n_species)
g_RT = h_RT - s0_R
kr = rand(gas.n_reactions);

In [10]:
function kr!(kr, kf,g_RT,vk,vk_sum,T)
    mul!(kr,vk,g_RT) #kr proxies as Δg_RT as intermediate storage'
    logP = log(one_atm / R / T)
    @. kr = kf / exp(-kr + logP * vk_sum)
    return nothing
end

function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,reversibleReactions::ReversibleReactions,g_RT::AbstractVector ,T::Real)
    forwardRate!(kf, reversibleReactions.arrhenius_coeffs,T)
    kr!(kr,kf,g_RT,reversibleReactions.vk,reversibleReactions.vk_sum,T)
    kf_Pi_c_rectantOrder!(kf,C,reversibleReactions.reactant_orders)
    kf_Pi_c_rectantOrder!(kr,C,reversibleReactions.product_stoich_coeffs)
    kf .-= kr
    vk_kf!(wdot,kf,reversibleReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$kr,$C,$reversibleReactions,$g_RT,$T) #evals = 200 seconds = 20

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  14.551 μs … 130.195 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     14.829 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   15.780 μs ±   5.454 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▅▃▁                                                         ▁
  ██████▇▇▆▇▆▆▅▆▅▇▆▆▅▆▅▅▅▅▆▆▅▄▅▃▄▅▁▄▆▅▄▄▄▅▅▅▃▅▄▃▅▄▅▅▅▅▄▅▅▄▅▄▅▄ █
  14.6 μs       Histogram: log(frequency) by time      43.4 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [60]:
function wdot_func(reaction, T, C,h_RT,s0_R)

    @inbounds _kf = @. @view(reaction.Arrhenius_coeffs[:, 1]) * exp(
        @view(reaction.Arrhenius_coeffs[:, 2]) * log(T) -
        @view(reaction.Arrhenius_coeffs[:, 3]) * (4184.0 / R / T),
    )

    @inbounds ΔS_R = reaction.vk' * s0_R 
    @inbounds ΔH_RT = reaction.vk' * h_RT
    @inbounds Keq =
        @. exp(ΔS_R - ΔH_RT + log(one_atm / R / T) * reaction.vk_sum)
    @inbounds _kr = @. _kf / Keq * reaction.is_reversible

    for i = 1:reaction.n_reactions
        @inbounds for j in reaction.i_reactant[i]
            @inbounds _kf[i] *= C[j]^reaction.reactant_orders[j, i]
        end
        @inbounds for j in reaction.i_product[i]
            @inbounds _kr[i] *= C[j]^reaction.product_stoich_coeffs[j, i]
        end
    end
    #if get_qdot
    #    return _kf - _kr
    #end
    return reaction.vk *  (_kf -_kr ) #omega_dot_k
end
@benchmark wdot_func($gas.kinetics,$T,$C,$h_RT,$s0_R)

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  33.176 μs …   9.221 ms  ┊ GC (min … max): 0.00% … 98.95%
 Time  (median):     34.391 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   40.606 μs ± 138.163 μs  ┊ GC (mean ± σ):  5.78% ±  1.71%

  ██▅▄▃▂▁▁▁                ▁▁▁   ▁                             ▂
  █████████████████▇▇█▇▇████████████▇▇▇▇▇▆▇▆▅▅▅▅▆▅▅▆▁▅▅▅▆▅▆▅▃▅ █
  33.2 μs       Histogram: log(frequency) by time      93.2 μs <

 Memory estimate: 16.62 KiB, allocs estimate: 8.

In [61]:
wdot .=0.0
wdot_func!(wdot,kf,kr,C,reversibleReactions,g_RT,T)
wdot-wdot_func((gas.kinetics),T,C,h_RT,s0_R)

DimensionMismatch: DimensionMismatch: 

# Three body reaction 

In [13]:
kf_tB = ones(12)
kr_tB = ones(12)
kf_tB_old = ones(12)


function efficency!( kf,effCoeff,C)
    for ir in eachindex(kf)
        dot_C_eff = 0.0
        @inbounds for j in effCoeff.colptr[ir]:effCoeff.colptr[ir+1]-1
            @inbounds dot_C_eff += effCoeff.nzval[j] * C[effCoeff.rowval[j]]
        end
        @inbounds kf[ir] *= dot_C_eff
    end
    return nothing
end

function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,threeBodyReactions::ThreeBodyReactions,g_RT::AbstractVector ,T::Real)
    forwardRate!(kf, threeBodyReactions.arrhenius_coeffs,T)
    efficency!(kf,threeBodyReactions.efficiency_coeffs,C)
    kr!(kr,kf,g_RT,threeBodyReactions.vk,threeBodyReactions.vk_sum,T)
    kf_Pi_c_rectantOrder!(kf,C,threeBodyReactions.reactant_orders)
    kf_Pi_c_rectantOrder!(kr,C,threeBodyReactions.product_stoich_coeffs)
    kf .-= kr
    vk_kf!(wdot,kf,threeBodyReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf_tB,$kr_tB,$C,$threeBodyReactions,$g_RT,$T) #evals = 200 seconds = 20

BenchmarkTools.Trial: 10000 samples with 10 evaluations.
 Range (min … max):  1.120 μs …  12.217 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.146 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.188 μs ± 383.637 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ██▄▂▁                                                       ▂
  █████▇▇▅▅▅▅▅▆▅▃▅▅▅▄▆▆▃▅▄▅▅▅▁▆▅▅▁▃▁▃▁▄▁▄▁▅▃▄▁▁▃▃▄▁▁▃▁▁▄▄▁▃▁▃ █
  1.12 μs      Histogram: log(frequency) by time      2.53 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

# fallOff

In [15]:
struct FallOffReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_sum::vTy
    efficiency_coeffs::sTy
    Arrhenius_0::Ty
end

i_fO = Int64[]
i_troe = Int64[]
for (i, index) in enumerate(gas.kinetics.index_falloff )
    if gas.kinetics.index_falloff_Troe[i] < 0
        push!(i_fO,index)
    else
        push!(i_troe,index)
    end
end

fallOffReactions = FallOffReactions(
    gas.kinetics.Arrhenius_coeffs[i_fO,:],
    gas.kinetics.reactant_orders[:,i_fO],
    gas.kinetics.product_stoich_coeffs[:,i_fO],
    gas.kinetics.reactant_stoich_coeffs[:,i_fO],
    sparse(transpose(gas.kinetics.vk[:,i_fO])),
    gas.kinetics.vk_sum[i_fO],
    gas.kinetics.efficiencies_coeffs[:,i_fO],
    gas.kinetics.Arrhenius_0
    );

kf_fO = zeros(length(i_fO))
kr_fO = zeros(length(i_fO));

kf_troe = zeros(length(i_troe))
kr_troe = zeros(length(i_troe));

In [16]:

function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,fallOffReactions::FallOffReactions,g_RT::AbstractVector ,T::Real)
    forwardRate!(kf, fallOffReactions.arrhenius_coeffs,T)    
    #fallOff adaptation
    forwardRate!(kr,fallOffReactions.Arrhenius_0,T) #kr proxies as k0
    efficency!(kr,fallOffReactions.efficiency_coeffs,C) #kr proxies as k0 * dot(eff,C)
    @inbounds kr ./= kf #kr proxies as Pr
    @inbounds @. kf *= (kr / (1 + kr))
    kr!(kr,kf,g_RT,fallOffReactions.vk,fallOffReactions.vk_sum,T)
    kf_Pi_c_rectantOrder!(kf,C,fallOffReactions.reactant_orders)
    kf_Pi_c_rectantOrder!(kr,C,fallOffReactions.product_stoich_coeffs)
    kf .-= kr
    vk_kf!(wdot,kf,fallOffReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf_fO,$kr_fO,$C,$fallOffReactions,$g_RT,$T)

BenchmarkTools.Trial: 10000 samples with 195 evaluations.
 Range (min … max):  486.210 ns …  1.989 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     490.644 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   513.870 ns ± 80.616 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▅▃▂▁                                                        ▁
  ██████████▇▇▇▇▇▆▅▆▇▇▇▆▇▆▆▇▆▇▆▆▅▅▆▆▅▆▆▅▆▅▅▅▄▅▄▄▆▄▅▅▄▅▅▅▅▄▄▄▄▄ █
  486 ns        Histogram: log(frequency) by time       903 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

# Troe reactions 

In [18]:
struct TroeReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_sum::vTy
    efficiency_coeffs::sTy
    Arrhenius_0::Ty
    troe::Ty
end

troeReactions = TroeReactions(
    gas.kinetics.Arrhenius_coeffs[i_troe,:],
    gas.kinetics.reactant_orders[:,i_troe],
    gas.kinetics.product_stoich_coeffs[:,i_troe],
    gas.kinetics.reactant_stoich_coeffs[:,i_troe],
    sparse(transpose(gas.kinetics.vk[:,i_troe])),
    gas.kinetics.vk_sum[i_troe],
    gas.kinetics.efficiencies_coeffs[:,i_troe],
    gas.kinetics.Arrhenius_0,
    gas.kinetics.Troe_
    );

In [66]:
function troe!(kf, logPr,troeCoeffs,T)
  for i in eachindex(kf)
    @inbounds F_cent =
                (1 - troeCoeffs[i, 1]) * exp(-T / troeCoeffs[i, 4]) +
                troeCoeffs[i, 1] * exp(-T / troeCoeffs[i, 2]) +
                exp(-troeCoeffs[i, 3] / T)
    lF_cent = log10(F_cent)
    _C = -0.4 - 0.67 * lF_cent
    N = 0.75 - 1.27 * lF_cent
    f1 = (logPr[i] + _C) / (N - 0.14 * (logPr[i] + _C))
    @inbounds kf[i] *= exp(log(10.0) * lF_cent / (1 + f1^2))
  end
end

function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,troeReactions::TroeReactions,g_RT::AbstractVector ,T::Real)
    forwardRate!(kf, troeReactions.arrhenius_coeffs,T)
    
    #fallOff adaptation
    forwardRate!(kr,troeReactions.Arrhenius_0,T) #kr proxies as k0
    efficency!(kr,troeReactions.efficiency_coeffs,C) #kr proxies as k0 * dot(eff,C)
    @inbounds kr ./= kf #kr proxies as Pr
    @inbounds @. kf *= (kr / (1 + kr))
    @inbounds kr .= log10.(kr) #kr proxies as logPr
    troe!(kf,kr, troeReactions.troe,T)

    kr!(kr,kf,g_RT,troeReactions.vk,troeReactions.vk_sum,T)
    kf_Pi_c_rectantOrder!(kf,C,troeReactions.reactant_orders)
    kf_Pi_c_rectantOrder!(kr,C,troeReactions.product_stoich_coeffs)
    kf .-= kr
    vk_kf!(wdot,kf,troeReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf_troe,$kr_troe,$C,$troeReactions,$g_RT,$T)

BenchmarkTools.Trial: 10000 samples with 8 evaluations.
 Range (min … max):  3.848 μs … 22.153 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     3.885 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.183 μs ±  1.182 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▄▃▂▁                                                      ▁
  ██████▇▆▇▆▆▆▆▆▇▆▆▅▅▆▆▆▆▆▅▄▅▅▅▅▄▆▆▆▅▅▅▇▆▆▅▆▅▅▆▅▄▄▄▃▃▃▄▃▄▃▂▄ █
  3.85 μs      Histogram: log(frequency) by time     9.66 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

# Putting it all together 

In [20]:
struct TestKinetics{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}
    forwardReactions::ForwardReactions{Ty,sTy}
    reversibleReactions::ReversibleReactions{Ty,sTy,vTy}
    threeBodyReactions::ThreeBodyReactions{Ty,sTy,vTy}
    fallOffReactions::FallOffReactions{Ty,sTy,vTy}
    troeReactions::TroeReactions{Ty,sTy,vTy}
end

mutable struct State{vTy<:AbstractVector}
    kf_forward::vTy
    kf_reversible::vTy
    kr_reversible::vTy
    kf_threeBody::vTy
    kr_threeBody::vTy
    kf_fallOff::vTy
    kr_fallOff::vTy
    kf_troe::vTy
    kr_troe::vTy
    C::vTy
    s0_R::vTy
    h_RT::vTy
    g_RT::vTy
end

#state = State()


In [75]:
i_forward = .!(gas.kinetics.is_reversible)
i_rev = Int64[]
for (i) in 1:gas.n_reactions
    if !(i in  gas.kinetics.index_three_body) & !(i in  gas.kinetics.index_falloff) & gas.kinetics.is_reversible[i]
        push!(i_rev,i)
    end

end
i_tB = gas.kinetics.index_three_body
i_fO = Int64[]
i_troe = Int64[]
i_A0_fO = Int64[]
i_A0_troe = Int64[]
for (i, index) in enumerate(gas.kinetics.index_falloff )
    if gas.kinetics.index_falloff_Troe[i] < 0
        push!(i_fO,index)
        push!(i_A0_fO,i)
    else
        push!(i_troe,index)
        push!(i_A0_troe,i)
    end
end



In [84]:
forwardReactions = ForwardReactions(gas.kinetics.Arrhenius_coeffs[i_forward,:],
                    gas.kinetics.reactant_orders[:,i_forward],
                    sparse(transpose(gas.kinetics.vk[:,i_forward]))
                    )

reversibleReactions = ReversibleReactions(
    gas.kinetics.Arrhenius_coeffs[i_rev,:],
    gas.kinetics.reactant_orders[:,i_rev],
    gas.kinetics.product_stoich_coeffs[:,i_rev],
    gas.kinetics.reactant_stoich_coeffs[:,i_rev],
    sparse(transpose(gas.kinetics.vk[:,i_rev])),
    gas.kinetics.vk_sum[i_rev]
    );
threeBodyReactions = ThreeBodyReactions(
    gas.kinetics.Arrhenius_coeffs[i_tB,:],
    gas.kinetics.reactant_orders[:,i_tB],
    gas.kinetics.product_stoich_coeffs[:,i_tB],
    gas.kinetics.reactant_stoich_coeffs[:,i_tB],
    sparse(transpose(gas.kinetics.vk[:,i_tB])),
    gas.kinetics.vk_sum[i_tB],
    gas.kinetics.efficiencies_coeffs[:,i_tB]
    );
    
fallOffReactions = FallOffReactions(
    gas.kinetics.Arrhenius_coeffs[i_fO,:],
    gas.kinetics.reactant_orders[:,i_fO],
    gas.kinetics.product_stoich_coeffs[:,i_fO],
    gas.kinetics.reactant_stoich_coeffs[:,i_fO],
    sparse(transpose(gas.kinetics.vk[:,i_fO])),
    gas.kinetics.vk_sum[i_fO],
    gas.kinetics.efficiencies_coeffs[:,i_fO],
    gas.kinetics.Arrhenius_0[i_A0_fO,:]
    );

troeReactions = TroeReactions(
    gas.kinetics.Arrhenius_coeffs[i_troe,:],
    gas.kinetics.reactant_orders[:,i_troe],
    gas.kinetics.product_stoich_coeffs[:,i_troe],
    gas.kinetics.reactant_stoich_coeffs[:,i_troe],
    sparse(transpose(gas.kinetics.vk[:,i_troe])),
    gas.kinetics.vk_sum[i_troe],
    gas.kinetics.efficiencies_coeffs[:,i_troe],
    gas.kinetics.Arrhenius_0[i_A0_troe,:],
    gas.kinetics.Troe_
    );

kinetics = TestKinetics(
    forwardReactions,reversibleReactions,threeBodyReactions,fallOffReactions,troeReactions
)

state = State(
    zeros(sum(i_forward)),
    zeros(length(i_rev)),
    zeros(length(i_rev)),
    zeros(length(i_tB)),
    zeros(length(i_tB)),
    zeros(length(i_fO)),
    zeros(length(i_fO)),
    zeros(length(i_troe)),
    zeros(length(i_troe)),
    C,
    s0_R,
    h_RT,
    g_RT,
)

State{Vector{Float64}}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.006952637777283083, 0.017900852956448717, 0.03269067031170752, 0.036505427744370586, 0.016112220711592587, 0.03233130986293409, 0.002883986365450892, 0.034924619246653424, 0.03526846241587904, 0.01575888045394774  …  0.013806441081698772, 0.004936971248167175, 0.012867964480641907, 0.022745521608903033, 0.0260288

In [85]:
function wdot_func!(wdot,state,kinetics ,T::Real)
    wdot .= 0.0
    wdot_func!(wdot,state.kf_forward,state.C,kinetics.forwardReactions ,T)
    wdot_func!(wdot,state.kf_reversible,state.kf_reversible,state.C,kinetics.reversibleReactions,state.g_RT,T)
    wdot_func!(wdot,state.kf_threeBody,state.kr_threeBody,state.C,kinetics.threeBodyReactions,state.g_RT,T)
    wdot_func!(wdot,state.kf_fallOff,state.kr_fallOff,state.C,kinetics.fallOffReactions,state.g_RT,T)
    wdot_func!(wdot,state.kf_troe,state.kr_troe,state.C,kinetics.troeReactions,state.g_RT,T)
    return nothing
end
@benchmark wdot_func!($wdot,$state,$kinetics,$T) evals=20

BenchmarkTools.Trial: 10000 samples with 20 evaluations.
 Range (min … max):  17.881 μs … 221.468 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     20.068 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   22.114 μs ±   6.215 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

   █▅▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁ ▁                           ▂
  ▂██████████████████████████████████████▇▇█▇▇▇▆▆▇▇▇▆▆▅▆▆▆▄▆▆▅ █
  17.9 μs       Histogram: log(frequency) by time      39.4 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [86]:
@benchmark wdot_func($gas.kinetics,$T,$C,$s0_R,$h_RT) evals=20

BenchmarkTools.Trial: 5597 samples with 20 evaluations.
 Range (min … max):  33.051 μs …  2.883 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     38.671 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   44.100 μs ± 52.686 μs  ┊ GC (mean ± σ):  6.63% ± 7.57%

  ▆█▁                                                          
  ███▅▆▆▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂ ▃
  33.1 μs         Histogram: frequency by time        71.8 μs <

 Memory estimate: 16.62 KiB, allocs estimate: 8.

In [87]:
wdot_func!(wdot,state,kinetics,T)
wdot - Arrhenius.wdot_func(gas.kinetics,T,C,s0_R,h_RT)

53-element Vector{Float64}:
      -2.9415156739826947e7
      -3.3424398628827393e8
       7.653153644204879e8
      -1.2381317740843102e8
      -6.468561501188469e7
       1.5602806020065123e8
      -1.290718131702278e8
       2.2218755647286517e6
       6.548658085037045e6
      -1.5381510378865984e8
       ⋮
 -859353.39700884
       1.8550704546884452e6
      -5.346053062090595e6
       1.2962225759134474e8
       0.0
       1.3298239327978663e8
      -6.4997034670402985e6
      -4.0227335877694026e7
      -2.095420884573522e7

In [47]:
@benchmark calc_omegaY!( $wdot,$gas, $T, $state.C, $state.s0_R, $state.h_RT,$(1.2)) 

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  178.002 μs …  2.116 ms  ┊ GC (min … max): 0.00% … 83.96%
 Time  (median):     185.968 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   213.127 μs ± 72.989 μs  ┊ GC (mean ± σ):  0.77% ±  2.68%

  ▇█▆▄▃▂▂▃▃▃▃▂▂▃▃▃▂▂▂▂▁▁▁▁▁▁ ▁▁ ▁▁▁                            ▂
  ███████████████████████████████████████▇▇▇▇█▇▇▇▆▆▆▆▆▇▆▅▆▅▅▅▅ █
  178 μs        Histogram: log(frequency) by time       398 μs <

 Memory estimate: 46.80 KiB, allocs estimate: 650.

# Comparison to unified framework 


In [35]:
mutable struct UnifiedState{vTy<:AbstractVector}
    kf::vTy
    kr::vTy
    C::vTy
    s0_R::vTy
    h_RT::vTy
    g_RT::vTy
end

unified_state= UnifiedState(
    zeros(gas.n_reactions),
    zeros(gas.n_reactions),
    C,
    s0_R,
    h_RT,
    g_RT
)

UnifiedState{Vector{Float64}}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.5870055255522351, 0.700508010312609, 0.9236726738619351, 0.8474256750537372, 0.10723819435190884, 0.2397152844410113, 0.17322521185755702, 0.3963359177601694, 0.6788876751137365, 0.844964054437262  …  0.8754944336002816, 0.7856169488021072, 0.8325481887758279, 0.7573173556225734, 0.4402973313847178, 0.9828391073619709, 0.24921386352191943, 0.11713318415645513, 0.5875903121505678, 0.30905624069895055], [0.30596799747059955, 0.8176516715856815, 0.4615040864657842, 0.8123114101120608, 0.5360047712116782, 0.23333573357231596, 0.8903004691872667, 0.9521438509290644, 0.3074812643992122, 0.062439422093573116  …  0.4618643336253432, 0.8945724371657331, 0.18812474084916342, 0.19882164603387487, 0.6914742239999521, 0.3045330093369266, 0.3657595592087247, 0.

In [36]:
struct UnifiedReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_rev::sTy
    vk_sum::vTy
    efficiency_coeffs_tB::sTy
    efficiency_coeffs_fO::sTy
    Arrhenius_0::Ty
    troe::Ty
    index_rev_start::Int64
    index_three_body_start::Int64
    index_fallOff_start::Int64
    index_troe_start::Int64
end



In [37]:
arr_coeffs = vcat(
    forwardReactions.arrhenius_coeffs,
    reversibleReactions.arrhenius_coeffs,
    threeBodyReactions.arrhenius_coeffs,
    fallOffReactions.arrhenius_coeffs,
    troeReactions.arrhenius_coeffs)

reactant_orders = hcat(
    forwardReactions.reactant_orders,
    reversibleReactions.reactant_orders,
    threeBodyReactions.reactant_orders,
    fallOffReactions.reactant_orders,
    troeReactions.reactant_orders)

product_stoich_coeffs = hcat(
    reversibleReactions.product_stoich_coeffs,
    threeBodyReactions.product_stoich_coeffs,
    fallOffReactions.product_stoich_coeffs,
    troeReactions.product_stoich_coeffs)

reactant_stoich_coeffs = hcat(
        reversibleReactions.reactant_stoich_coeffs,
        threeBodyReactions.reactant_stoich_coeffs,
        fallOffReactions.reactant_stoich_coeffs,
        troeReactions.reactant_stoich_coeffs)

vk = vcat(
            forwardReactions.vk,
            reversibleReactions.vk,
            threeBodyReactions.vk,
            fallOffReactions.vk,
            troeReactions.vk)

vk_rev = vcat(
    reversibleReactions.vk,
    threeBodyReactions.vk,
    fallOffReactions.vk,
    troeReactions.vk)

vk_sum = vcat(
    reversibleReactions.vk_sum,
    threeBodyReactions.vk_sum,
    fallOffReactions.vk_sum,
    troeReactions.vk_sum
)

efficiency_coeffs_tB=threeBodyReactions.efficiency_coeffs

efficiency_coeffs_fO = hcat(
    fallOffReactions.efficiency_coeffs,
    troeReactions.efficiency_coeffs)

Arrhenius_0 =hcat(
    fallOffReactions.Arrhenius_0,
    troeReactions.Arrhenius_0)

troe = troeReactions.troe

index_rev_start = sum(i_forward)+1
index_three_body_start = index_rev_start + length(i_rev) 
index_fallOff_start = index_three_body_start + length(i_tB) 
index_troe_start = index_fallOff_start + length(i_fO) 
index_troe_start+length(i_troe)


326

In [38]:
unifiedReactions = UnifiedReactions(
    arr_coeffs,
    reactant_orders,
    product_stoich_coeffs,
    reactant_stoich_coeffs,
    vk,
    vk_rev,
    vk_sum,
    efficiency_coeffs_tB,
    efficiency_coeffs_fO,
    Arrhenius_0,
    troe,
    index_rev_start,
    index_three_body_start,
    index_fallOff_start,
    index_troe_start,
)

UnifiedReactions{Matrix{Float64}, SparseMatrixCSC{Float64, Int64}, Vector{Float64}}([5.0e9 0.0 1500.0; 3.37e10 0.0 0.0; … ; 2550.0 1.6 5700.0; 3.613e10 0.0 0.0], sparse([4, 11, 3, 13, 5, 13, 4, 11, 11, 6  …  1, 10, 2, 29, 13, 26, 13, 25, 2, 50], [1, 1, 2, 2, 3, 3, 4, 4, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19, 19, 20, 20, 21, 21, 22, 22, 23, 23, 24, 24, 25, 25, 26, 26, 27, 27, 28, 28, 29, 29, 30, 30, 31, 31, 32, 32, 33, 33, 34, 34, 35, 35, 36, 36, 37, 37, 38, 38, 39, 39, 40, 40, 41, 41, 42, 42, 43, 43, 44, 44, 45, 45, 46, 46, 47, 47, 47, 48, 48, 48, 49, 49, 49, 50, 50, 51, 51, 52, 52, 53, 53, 54, 54, 55, 55, 56, 56, 57, 57, 58, 58, 59, 59, 60, 60, 61, 61, 62, 62, 63, 63, 64, 64, 65, 65, 66, 66, 67, 67, 68, 68, 69, 69, 70, 70, 71, 71, 72, 72, 73, 73, 74, 74, 75, 75, 76, 76, 77, 77, 78, 78, 79, 79, 80, 80, 81, 81, 82, 83, 83, 84, 84, 85, 85, 86, 86, 87, 87, 88, 88, 89, 89, 90, 90, 91, 91, 92, 92, 93, 93, 94, 94, 95, 95, 96, 96

In [42]:
function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,g_RT::AbstractVector,reactions::UnifiedReactions ,T::Real)

    forwardRate!(kf, reactions.arrhenius_coeffs,T)

    #threeBody
    tB_range = reactions.index_three_body_start:reactions.index_fallOff_start-1
    efficency!(@view(kf[tB_range]),reactions.efficiency_coeffs_tB,C)

    #fallOff adaptation
    fO_start=  reactions.index_fallOff_start#:end#reactions.index_fallOff_start-1
    forwardRate!(@view(kr[fO_start:end]),reactions.Arrhenius_0,T) #kr proxies as k0
    efficency!(@view(kr[fO_start:end]),reactions.efficiency_coeffs_fO,C) #kr proxies as k0 * dot(eff,C)
    @inbounds @view(kr[fO_start:end]) ./= @view(kf[fO_start:end]) #kr proxies as Pr
    @inbounds @. @view(kf[fO_start:end]) *= (@view(kr[fO_start:end]) / (1 + @view(kr[fO_start:end])))

    #troe
    troe_start = reactions.index_troe_start
    @inbounds kr[troe_start:end] .= log10.(@view(kr[troe_start:end])) #kr proxies as logPr
    troe!(@view(kf[troe_start:end]),@view(kr[troe_start:end]), reactions.troe,T)


    kr!(@view(kr[reactions.index_rev_start:end]),@view(kf[reactions.index_rev_start:end]),g_RT,reactions.vk_rev,reactions.vk_sum,T)
    kf_Pi_c_rectantOrder!(kf,C,reactions.reactant_orders)
    kf_Pi_c_rectantOrder!(@view(kr[reactions.index_rev_start:end]),C,reactions.product_stoich_coeffs)
    kf .-=kr
    vk_kf!(wdot,kf,reactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$unified_state.kf,$unified_state.kr,$unified_state.C,$unified_state.g_RT,$unifiedReactions,$T) evals=20

BenchmarkTools.Trial: 10000 samples with 20 evaluations.
 Range (min … max):  17.657 μs … 56.730 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     18.679 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   20.644 μs ±  3.919 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▆▄▃▃▃▃▃▃▃▃▃▃▃▃▂▂▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁ ▁                         ▂
  ███████████████████████████████████████▇██▇▇▇▆▆▆▇▆▆▆▆▆▅▄▅▅▅ █
  17.7 μs      Histogram: log(frequency) by time      34.9 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [43]:
@benchmark wdot_func!($wdot,$state,$kinetics,$T) evals=20

BenchmarkTools.Trial: 10000 samples with 20 evaluations.
 Range (min … max):  18.334 μs … 46.192 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     18.570 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   20.027 μs ±  2.850 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▅▄▃▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁                                 ▁
  █████████████████████████████████▇█▇██▇▇▇▇▇▇▇▆▇▅▆▆▄▆▆▅▆▆▅▄▄ █
  18.3 μs      Histogram: log(frequency) by time      30.6 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [74]:
wdot_old = wdot_func(gas.kinetics,T,C,s0_R,h_RT)
wdot2 = copy(wdot)
calc_omegaY!( wdot2,gas, T, state.C, state.s0_R, state.h_RT,(1.0)) 
wdot_uni = copy(wdot)
wdot_func!(wdot_uni,unified_state.kf,unified_state.kr,unified_state.C,unified_state.g_RT,unifiedReactions,T)
wdot_func!(wdot,state,kinetics,T)

println( wdot2[1] )
println( wdot_old[1] )
println( wdot[1] )
println( wdot_uni[1] )

-1.3027737185410832e11
2.0480150426238477e8
4.072216525542138e10
-9.001633474918945e10
